# Object detection with YOLO11: from the batch app to an interactive notebook

This notebook is the interactive half of a pair. Its sibling,
[`app-object-detection`](https://github.com/OneSciencePlace/app-object-detection),
runs the same detection as a **batch** container: one `run.py` invocation, one
image in, an annotated JPEG and a `detections.json` out.

Here the *same logic* is unrolled into executable cells. Nothing is imported
from `run.py` and nothing is shelled out to it — the point is to see each step,
change it, and rerun it.

You will:

1. map the command-line arguments of `run.py` onto notebook variables and widgets,
2. validate an input image and choose an output directory,
3. load the YOLO11 weights that are **baked into this image**,
4. run inference at a confidence threshold,
5. write `<stem>_detected.jpg` and `detections.json` — byte-for-byte the batch app's outputs,
6. print the same human-readable summary the Tapis job log would show, and
7. rerun inference from a slider to see what the `--conf` flag actually does.

> **Where things live.** The model is baked in at `/app/yolo11n.pt` (read it
> from `APP_MODEL`, never hardcode it). This notebook was copied into your
> session working directory at startup, and every file it reads or writes is
> **relative to that directory** — the same contract `run.py` follows. Under
> Docker that directory is `/home/ospuser`; under Apptainer it is whatever
> directory you ran from.

## 1. How the script's command line maps onto a notebook

`run.py` takes four arguments:

```python
p.add_argument("--image", required=True)
p.add_argument("--output-dir", default=".")
p.add_argument("--model", default=os.environ.get("APP_MODEL", "/app/yolo11n.pt"))
p.add_argument("--conf", type=float, default=0.25)
```

`argparse` is wrong here — in a notebook it would try to parse Jupyter's own
command line. So the four settings become ordinary Python values, two of them
attached to widgets. The defaults below are the script's defaults, unchanged.

| `run.py` | here |
|---|---|
| `--image photo.jpg` | `image_path_text` (a text box, seeded from the first image found) |
| `--output-dir .` | `OUTPUT_DIR`, the working directory |
| `--model $APP_MODEL` | `MODEL_PATH`, read from the same env var |
| `--conf 0.25` | `INITIAL_CONFIDENCE`, and later the slider |

## 2. Imports

The script's imports, plus the display and widget helpers a notebook wants.
`cv2` arrives with ultralytics (which depends on `opencv-python`), which is why
the image carries `libgl1`.

In [ ]:
import json
import os
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import HTML, JSON, clear_output, display
from ultralytics import YOLO

%matplotlib inline

## 3. Configuration: the four arguments, as values

**The model.** `APP_MODEL` is set to `/app/yolo11n.pt` in the image, and the
weights were downloaded at *build* time. That is deliberate: a `.sif` is mounted
read-only and HPC compute nodes are frequently offline, so nothing may be
fetched on first use. Passing a bare `"yolo11n.pt"` to `YOLO()` would make
ultralytics try to download it — fine on a laptop, a hang or a crash on a
compute node.

**Everything else is relative to the working directory.** `run.py` reads and
writes relative paths because Tapis stages job inputs into the working
directory, and because Apptainer ignores the image's `WORKDIR` and uses the
caller's directory instead. This notebook does the same, so an image sitting
next to the notebook is found by its bare filename.

In [ ]:
WORK_DIR = Path.cwd()
OUTPUT_DIR = WORK_DIR          # run.py's --output-dir default is "."

# Same default, same env var, as run.py's --model.
MODEL_PATH = os.environ.get("APP_MODEL", "/app/yolo11n.pt")
INITIAL_CONFIDENCE = 0.25      # run.py's --conf default

print(f"Working directory: {WORK_DIR}")
print(f"Model:             {MODEL_PATH}")

if not Path(MODEL_PATH).is_file():
    print(
        f"\nNOTE: {MODEL_PATH} is missing. Inside this container it is baked in;\n"
        "outside it, set APP_MODEL to a local .pt file. Ultralytics would\n"
        "otherwise try to download weights, which a compute node cannot do."
    )

## 4. Choose an image

The batch app is handed its image by Tapis. Here you either point at a file
already in the working directory or upload one, which is written into that same
directory — so the notebook, the input, and the outputs all sit together, and a
bind-mounted working directory keeps all three after the container exits.

In [ ]:
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

existing = sorted(
    p for p in WORK_DIR.iterdir()
    if p.is_file()
    and p.suffix.lower() in IMAGE_SUFFIXES
    and not p.stem.endswith("_detected")     # don't pick up our own output
)
default_image = existing[0] if existing else WORK_DIR / "photo.jpg"

image_path_text = widgets.Text(
    value=str(default_image),
    description="Image path:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "initial"},
)
upload = widgets.FileUpload(accept="image/*", multiple=False, description="Upload image")
upload_status = widgets.HTML(value="")


def _save_upload(change):
    if not upload.value:
        return
    uploaded = upload.value[0]           # ipywidgets 8 returns a tuple of records
    destination = WORK_DIR / Path(uploaded["name"]).name   # .name strips any path
    destination.write_bytes(bytes(uploaded["content"]))
    image_path_text.value = str(destination)
    upload_status.value = f"<b>Saved:</b> {destination.name}"


upload.observe(_save_upload, names="value")
display(image_path_text, widgets.HBox([upload, upload_status]))

if existing:
    print(f"Found {len(existing)} image(s) here; using {default_image.name} unless you change it.")
else:
    print("No image found in the working directory — upload one, or edit the path above.")

## 5. Validate the path, create the output directory

`run.py` prints to stderr and calls `sys.exit(1)` when the image is missing.
A notebook should not kill its own kernel over a typo, so the same check raises
`FileNotFoundError` instead: fix the path in the widget above and rerun.

In [ ]:
image_path = Path(image_path_text.value).expanduser()
if not image_path.is_file():
    raise FileNotFoundError(
        f"Input image not found: {image_path}. "
        "Upload an image or edit the Image path field above, then rerun this cell."
    )

out_dir = OUTPUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Input image:      {image_path}")
print(f"Output directory: {out_dir}")

## 6. Load the model

`run.py` imports `YOLO` inside `main()` so that `--help` does not pay for the
torch import. In a notebook the import already happened, so the equivalent line
is just the constructor.

`yolo11n.pt` is the nano size, trained on COCO's 80 classes. Swapping in
`s`/`m`/`l`/`x`, or a different generation, changes only the weights file — the
API below is identical. In this image, though, only the baked-in file is
available offline.

In [ ]:
print(f"Loading model: {MODEL_PATH}")
model = YOLO(MODEL_PATH)
print(f"Classes the model knows: {len(model.names)}")

## 7. Run inference

The load-bearing two lines of `run.py`:

```python
results = model.predict(source=str(image_path), conf=args.conf, verbose=False)
result = results[0]
```

`predict` accepts a batch, so it returns a list; we pass one image and take
`results[0]`. `conf` drops every detection the model scores below the
threshold — the filtering happens inside `predict`, not afterwards.

In [ ]:
confidence = INITIAL_CONFIDENCE
print(f"Running inference on: {image_path} (conf={confidence})")

results = model.predict(source=str(image_path), conf=confidence, verbose=False)
result = results[0]

print(f"Detections kept at confidence >= {confidence:.2f}: {len(result.boxes)}")

## 8. Annotated image — the batch app's first output

`result.plot()` returns a NumPy array with the boxes and labels already drawn,
in OpenCV's BGR channel order, so `cv2.imwrite` writes it directly. Matplotlib
expects RGB, hence the `cvtColor` for the on-screen copy only.

The filename is `run.py`'s: `<stem>_detected.jpg`.

In [ ]:
annotated = result.plot()                     # NumPy array, BGR
out_image_path = out_dir / f"{image_path.stem}_detected.jpg"

if not cv2.imwrite(str(out_image_path), annotated):
    raise OSError(f"Failed to write annotated image: {out_image_path}")
print(f"Wrote annotated image: {out_image_path}")

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"YOLO11 detections, confidence >= {confidence:.2f}")
plt.show()

## 9. Structured detections

For each `box` in `result.boxes`, `run.py` pulls out:

- `box.cls[0]` → the class id, and `result.names[id]` → its name,
- `box.conf[0]` → the score,
- `box.xyxy[0]` → `[x_min, y_min, x_max, y_max]` in pixels.

The rounding below matches the script exactly, so the two produce identical
records for the same image and threshold.

In [ ]:
detections = []
for box in result.boxes:
    cls_id = int(box.cls[0])
    detections.append({
        "class": result.names[cls_id],
        "confidence": round(float(box.conf[0]), 4),
        "bbox_xyxy": [round(v, 1) for v in box.xyxy[0].tolist()],
    })

print(f"Structured {len(detections)} detection record(s).")
display(JSON(detections))

## 10. `detections.json` — the batch app's second output

Same three keys, same order, same file name. Anything downstream that consumes
the batch app's output consumes this notebook's output without knowing which
produced it — which is the whole point of keeping the two consistent.

In [ ]:
out_json_path = out_dir / "detections.json"
payload = {
    "image": image_path.name,
    "count": len(detections),
    "detections": detections,
}

with open(out_json_path, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Wrote detections: {out_json_path}")

## 11. The summary line

The tail of `run.py`, written for whoever reads the Tapis job log.

In [ ]:
if detections:
    print(f"Found {len(detections)} object(s):")
    for d in detections:
        print(f"  - {d['class']} ({d['confidence']:.0%})")
else:
    print("No objects detected above the confidence threshold.")

## 12. What the batch app cannot show you: sweeping `--conf`

Everything above reproduces a single `run.py` invocation. This is the part that
only works interactively.

The slider reruns the **same `model.predict` call** with a new `conf`. It does
not hide boxes after the fact — so what you see is exactly what
`run.py --conf <value>` would have written. Drag and release;
`continuous_update=False` keeps a full inference off every mouse movement.

In [ ]:
confidence_slider = widgets.FloatSlider(
    value=INITIAL_CONFIDENCE,
    min=0.05, max=0.95, step=0.05,
    description="Confidence:",
    readout_format=".2f",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="70%"),
)
interactive_output = widgets.Output()


def show_detection_at_confidence(threshold):
    current_path = Path(image_path_text.value).expanduser()
    if not current_path.is_file():
        raise FileNotFoundError(f"Input image not found: {current_path}")

    current_result = model.predict(
        source=str(current_path), conf=float(threshold), verbose=False
    )[0]

    current_detections = [
        {
            "class": current_result.names[int(box.cls[0])],
            "confidence": round(float(box.conf[0]), 4),
            "bbox_xyxy": [round(v, 1) for v in box.xyxy[0].tolist()],
        }
        for box in current_result.boxes
    ]

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(cv2.cvtColor(current_result.plot(), cv2.COLOR_BGR2RGB))
    ax.axis("off")
    ax.set_title(f"{len(current_detections)} detection(s) at confidence >= {threshold:.2f}")
    plt.show()

    if current_detections:
        rows = "".join(
            f"<tr><td>{d['class']}</td><td>{d['confidence']:.1%}</td>"
            f"<td><code>{d['bbox_xyxy']}</code></td></tr>"
            for d in current_detections
        )
        display(HTML(
            "<table><thead><tr><th>Class</th><th>Confidence</th><th>bbox_xyxy</th>"
            "</tr></thead><tbody>" + rows + "</tbody></table>"
        ))
    else:
        display(HTML("<p><b>No detections above this threshold.</b></p>"))


def _refresh_interactive_output(change=None):
    with interactive_output:
        clear_output(wait=True)
        try:
            show_detection_at_confidence(confidence_slider.value)
        except Exception as exc:
            print(f"{type(exc).__name__}: {exc}")


confidence_slider.observe(_refresh_interactive_output, names="value")
display(confidence_slider, interactive_output)
_refresh_interactive_output()

## 13. Reading the sweep

- **Lower thresholds keep more candidates**, including weak and wrong ones.
- **Higher thresholds keep fewer**, and lean on what the model is sure about.
- The threshold filters; it does not retrain. The weights never change.
- The right value is a property of your application, not of the model. A safety
  review may want recall and tolerate false positives; something that triggers
  an automated action wants precision.

Pick a value here, then pass it to the batch app as `--conf` for the full
dataset. That is the intended workflow for the pair: explore in the notebook,
run at scale in the batch container.

## 14. Running the batch app on the same image

The sibling container takes the value you settled on and produces the same two
files without a browser in the loop:

```bash
docker run --rm --user "$(id -u):$(id -g)" \
  -v "$PWD:/data" ghcr.io/onescienceplace/app-object-detection:latest \
  --image photo.jpg --conf 0.4

apptainer run app-object-detection-sif_latest.sif --image photo.jpg --conf 0.4
```

Both images bake in the same `yolo11n.pt` and write the same
`<stem>_detected.jpg` and `detections.json`, so results are comparable directly.

**Saving your work.** Anything you write lands in this notebook's directory.
Under Docker that survives the container only if you bind-mounted it
(`-v "$PWD:/home/ospuser"`); under Apptainer it is your own directory already.
One wrinkle inherited from `app-jupyter`: Jupyter's data directory lives under
`/tmp`, so the notebook-signing key is regenerated each session and saved
HTML/JS output is not trusted on reopen — rerun the cells, or point
`JUPYTER_DATA_DIR` somewhere persistent.